<a href="https://colab.research.google.com/github/bradleywjenks/wds-modelling-optimization-course/blob/main/notebooks/optimal_tank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimal tank operation

## Setup
Run the cells below first.
- **Google Colab:** the first cell clones (or updates) the repository. The first time, it also installs the `opwater` helper package and its dependencies, then restarts the session (Colab reports that the session crashed; this is expected). **Once it reconnects, run the first cell again**, then continue from the second cell. A pip warning about `numba` and `numpy` can be ignored.
- **Local:** install the repository first (see the README); the first cell then does nothing.

The second cell imports the packages used in this notebook.

In [ ]:
import sys

if "google.colab" in sys.modules:
    import importlib.util
    import os

    repo_dir = "/content/wds-modelling-optimization-course"
    if os.path.isdir(repo_dir):
        !git -C {repo_dir} pull --quiet --ff-only
    else:
        !git clone --depth 1 https://github.com/bradleywjenks/wds-modelling-optimization-course.git {repo_dir}

    # install opwater from the clone, unless this runtime already has it
    spec = importlib.util.find_spec("opwater")
    if spec is None or not spec.origin.startswith(repo_dir):
        %pip install --quiet -e {repo_dir}
        print("Restarting the session to load the new packages. Run this cell again once it reconnects.")
        os.kill(os.getpid(), 9)

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib_inline
import numpy as np
from IPython.display import Image

from opwater import data_dir

# Matplotlib parameters
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

plt.rc("font", size=10)  # controls default text sizes
plt.rc("axes", titlesize=10)  # fontsize of the axes title
plt.rc("axes", labelsize=11)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=10)  # fontsize of the tick labels
plt.rc("ytick", labelsize=10)  # fontsize of the tick labels
plt.rc("legend", fontsize=10)  # legend fontsize

## Problem formulation
We consider a tank operation problem using a simplified (and convex) model. The system has a single tank, a single inflow, and a single (aggregated) demand point, with hourly time steps $t \in \lbrace 0, \dots, T-1 \rbrace$.

- $c_t$ is the pumping/treatment cost of the tank inflow at time $t$ [per m$^3$] (**known**)
- $d_t$ is the water demand at time $t$ [m$^3$/h] (**known**)
- $S_{max}$ and $S_{min}$ are the maximum and minimum tank storage [m$^3$] (**known**)
- $S_{init}$ is the initial tank storage [m$^3$] (**known**)
- $I_{max}$ is the maximum tank inflow [m$^3$/h] (**known**)
- $S_t$ is the tank storage at time $t \in \lbrace 0, \dots, T \rbrace$ [m$^3$] (**state** variable)
- $I_t$ is the tank inflow at time $t$ [m$^3$/h] (**control** variable)

Load problem data.

In [ ]:
data = np.load(data_dir() / "optimal_tank" / "tank_operation_data.npz")

d = data["d"]  # water demand
c = data["c"]  # pumping/treatment cost
S_max = data["S_max"].item()  # max storage (tank capacity)
S_min = 0  # min storage
S_init = data["S_init"].item()  # initial storage
I_max = data["I_max"].item()  # max tank inflow
T = len(d)  # number of time steps

In [ ]:
Image(filename=str(data_dir() / "figures" / "tank_illustration.png"), width=150)

Our optimization problem aims to minimize the water pumping/treatment cost

$$\sum_{t=0}^{T-1} c_t I_t.$$

The problem is subject to mass balance constraints

$$S_{t+1} = S_t + I_t - d_t, \quad t = 0, \dots, T-1,$$

initial and final storage constraints

$$S_0 = S_{init}, \quad S_T = S_{init},$$

and physical constraints

$$S_{min} \leq S_t \leq S_{max}, \quad t = 0, \dots, T,$$
$$0 \leq I_t \leq I_{max}, \quad t = 0, \dots, T-1.$$

### CVXPY solver
We solve this problem using [CVXPY](https://www.cvxpy.org/).

In [ ]:
# define variables
S = cp.Variable(T + 1)  # storage
I = cp.Variable(T)  # inflow

# objective function
objective = cp.Minimize(c @ I)

# constraints
constraints = [S[0] == S_init, S[T] == S_init]  # initial and final storage

for t in range(T):
    constraints += [S[t + 1] == S[t] + I[t] - d[t]]  # mass balance
    constraints += [S[t + 1] >= S_min, S[t + 1] <= S_max]  # storage limits
    constraints += [I[t] >= 0, I[t] <= I_max]  # inflow limits

# solve problem
prob = cp.Problem(objective, constraints)
prob.solve()
print(f"Status: {prob.status}, optimal cost: {prob.value:.2f}")

Results plotting.

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(8, 6.5))

axs[0].plot(d, linewidth=1.5, color="black")
axs[0].set_yticks(np.arange(0, np.ceil(np.max(d) / 10) * 10 + 10, 10))
axs[0].set_xlabel("Time of day [h]")
axs[0].set_ylabel(r"Demand [$\mathrm{m}^3$/h]")

axs[1].plot(c, linewidth=1.5, color="black")
axs[1].set_yticks(np.arange(0, np.ceil(np.max(c) / 0.1) * 0.1 + 0.1, 0.1))
axs[1].set_xlabel("Time of day [h]")
axs[1].set_ylabel(r"Cost [per $\mathrm{m}^3$]")

axs[2].plot(S.value, linewidth=1.5, label="$S_t$", color="blue")
axs[2].plot(
    S_min * np.ones(T + 1), linewidth=1.5, label="$S_{min}$ / $S_{max}$", color="black", linestyle="--"
)
axs[2].plot(S_max * np.ones(T + 1), linewidth=1.5, color="black", linestyle="--")
axs[2].set_yticks(np.arange(0, np.ceil(S_max / 100) * 100 + 100, 200))
axs[2].set_xlabel("Time of day [h]")
axs[2].set_ylabel(r"$S_t$ [$\mathrm{m}^3$]")
leg = axs[2].legend(loc="right", edgecolor="k", borderpad=0.5, fancybox=False)
leg.get_frame().set_linewidth(0.5)

axs[3].plot(I.value, linewidth=1.5, label="$I_t$", color="blue")
axs[3].plot(I_max * np.ones(T), linewidth=1.5, label="$I_{max}$", color="black", linestyle="--")
axs[3].set_yticks(np.arange(0, np.ceil(I_max / 10) * 10 + 10, 10))
axs[3].set_xlabel("Time of day [h]")
axs[3].set_ylabel(r"$I_t$ [$\mathrm{m}^3$/h]")
leg = axs[3].legend(loc="right", edgecolor="k", borderpad=0.5, fancybox=False)
leg.get_frame().set_linewidth(0.5)

for ax in axs:
    ax.set_xlim([0, T])
    ax.set_ylim(bottom=0)
    ax.set_xticks(np.arange(0, T + 1, 4))
    ax.tick_params(axis="both", direction="in", length=4, width=0.6)
    ax.tick_params(right=True, top=True, labelright=False, labeltop=False, rotation=0)
    for axis in ["top", "bottom", "left", "right"]:
        ax.spines[axis].set_linewidth(0.5)

fig.tight_layout()
plt.show()

### Regularization
Our new objective is to minimize water pumping/treatment cost **and** changes in tank inflow $I_t$ between consecutive time steps. We include the latter objective using a regularization term with penalty parameter $\lambda \geq 0$, as follows

$$\sum_{t=0}^{T-1} c_t I_t + \lambda \sum_{t=0}^{T-2} |I_{t+1} - I_t|.$$

The same constraints as the first problem are applied.

Complete the code below to solve this problem using CVXPY.

In [ ]:
# define parameters
lam = 0.1  # penalty parameter

# define variables
S = ...
I = ...

# objective function
objective = ...

# constraints
constraints = ...

# solve problem
prob = ...

Results plotting.

In [ ]:
fig, axs = plt.subplots(5, 1, figsize=(8, 7.5))

axs[0].plot(d, linewidth=1.5, color="black")
axs[0].set_yticks(np.arange(0, np.ceil(np.max(d) / 10) * 10 + 10, 10))
axs[0].set_xlabel("Time of day [h]")
axs[0].set_ylabel(r"Demand [$\mathrm{m}^3$/h]")

axs[1].plot(c, linewidth=1.5, color="black")
axs[1].set_yticks(np.arange(0, np.ceil(np.max(c) / 0.1) * 0.1 + 0.1, 0.1))
axs[1].set_xlabel("Time of day [h]")
axs[1].set_ylabel(r"Cost [per $\mathrm{m}^3$]")

axs[2].plot(S.value, linewidth=1.5, label="$S_t$", color="blue")
axs[2].plot(
    S_min * np.ones(T + 1), linewidth=1.5, label="$S_{min}$ / $S_{max}$", color="black", linestyle="--"
)
axs[2].plot(S_max * np.ones(T + 1), linewidth=1.5, color="black", linestyle="--")
axs[2].set_yticks(np.arange(0, np.ceil(S_max / 100) * 100 + 100, 200))
axs[2].set_xlabel("Time of day [h]")
axs[2].set_ylabel(r"$S_t$ [$\mathrm{m}^3$]")
leg = axs[2].legend(loc="right", edgecolor="k", borderpad=0.5, fancybox=False)
leg.get_frame().set_linewidth(0.5)

axs[3].plot(I.value, linewidth=1.5, label="$I_t$", color="blue")
axs[3].plot(I_max * np.ones(T), linewidth=1.5, label="$I_{max}$", color="black", linestyle="--")
axs[3].set_yticks(np.arange(0, np.ceil(I_max / 10) * 10 + 10, 10))
axs[3].set_xlabel("Time of day [h]")
axs[3].set_ylabel(r"$I_t$ [$\mathrm{m}^3$/h]")
leg = axs[3].legend(loc="right", edgecolor="k", borderpad=0.5, fancybox=False)
leg.get_frame().set_linewidth(0.5)

I_delta = np.abs(np.diff(I.value, prepend=I.value[0]))
axs[4].plot(I_delta, linewidth=1.5, color="red")
axs[4].set_yticks(np.arange(0, np.ceil(np.max(I_delta) / 10) * 10 + 10, 10))
axs[4].set_xlabel("Time of day [h]")
axs[4].set_ylabel(r"$\Delta I_t$ [$\mathrm{m}^3$/h]")

for ax in axs:
    ax.set_xlim([0, T])
    ax.set_ylim(bottom=0)
    ax.set_xticks(np.arange(0, T + 1, 4))
    ax.tick_params(axis="both", direction="in", length=4, width=0.6)
    ax.tick_params(right=True, top=True, labelright=False, labeltop=False, rotation=0)
    for axis in ["top", "bottom", "left", "right"]:
        ax.spines[axis].set_linewidth(0.5)

fig.tight_layout()
plt.show()

## Discussion questions
1. What happens to the solution when we change the value of the penalty parameter $\lambda$?
2. What happens to the solution if we remove the final storage constraint $S_T = S_{init}$? Why is this constraint needed?
3. How does the optimal cost change if the tank capacity $S_{max}$ is increased or decreased? Why?

## References
- Boyd, S. and Vandenberghe, L. (2004). *Convex Optimization*. Cambridge University Press.
- Diamond, S. and Boyd, S. (2016). CVXPY: A Python-embedded modeling language for convex optimization. *Journal of Machine Learning Research*, 17(83), 1–5.